# YO G News — 04 Model Evaluation

This notebook performs a more detailed evaluation of the trained classifier.

**Evaluation includes**
- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix
- Example predictions


In [ ]:
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

DATA_PATH = Path("../dataset/news_dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../YO_G_News_Starter_Dataset.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("YO_G_News_Starter_Dataset.csv")

MODEL_PATH = Path("../backend/model/yog_news_model.pkl")

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        "Model not found. Run 03_Model_Training.ipynb first."
    )

df = pd.read_csv(DATA_PATH)
df["title"] = df["title"].fillna("").astype(str)
df["content"] = df["content"].fillna("").astype(str)
df["text"] = (df["title"] + " " + df["content"]).str.replace(r"\s+", " ", regex=True).str.strip()
df = df[df["text"].str.len() > 0].dropna(subset=["credibility_label"])
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

X = df["text"]
y = df["credibility_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model = joblib.load(MODEL_PATH)
predictions = model.predict(X_test)

print("Accuracy:", f"{accuracy_score(y_test, predictions) * 100:.2f}%")
print("Precision:", f"{precision_score(y_test, predictions, average='weighted', zero_division=0) * 100:.2f}%")
print("Recall:", f"{recall_score(y_test, predictions, average='weighted', zero_division=0) * 100:.2f}%")
print("F1-score:", f"{f1_score(y_test, predictions, average='weighted', zero_division=0) * 100:.2f}%")


In [ ]:
print(classification_report(y_test, predictions, zero_division=0))


In [ ]:
labels = model.classes_
cm = confusion_matrix(y_test, predictions, labels=labels)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(xticks_rotation=30)
plt.title("YO G News — Confusion Matrix")
plt.tight_layout()
plt.show()


In [ ]:
results = pd.DataFrame({
    "Actual": y_test.reset_index(drop=True),
    "Predicted": pd.Series(predictions)
})

display(results.head(15))


## Interpretation

A confusion matrix shows which credibility classes the model predicts correctly and which classes it confuses. Precision, recall, and F1-score are useful alongside accuracy, especially when the number of examples in each class is not equal.
